## Data Profiling

In [1]:
import pandas as pd
import sqlalchemy as sq
import pymysql as psql

print(sq.__version__)
print(pd.__version__)


2.0.49
2.3.3


In [2]:
from sqlalchemy import *
from pandas import *
from pymysql import *

from dotenv import load_dotenv
import os

load_dotenv()

user     = os.getenv("DB_USER")
pwd      = os.getenv("DB_PWD")
host     = os.getenv("DB_HOST")
port     = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{database}")


In [3]:
import pandas as pd

df = pd.read_sql("SELECT * FROM staging_transactions", engine)

print(df.shape)
print(df.dtypes)

(2512, 16)
transaction_id                       object
account_id                           object
transaction_amount                  float64
transaction_date             datetime64[ns]
transaction_type                     object
location                             object
device_id                            object
ip_address                           object
merchant_id                          object
channel                              object
customer_age                          int64
customer_occupation                  object
transaction_duration                  int64
login_attempts                        int64
account_balance                     float64
previous_transaction_date    datetime64[ns]
dtype: object


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   transaction_id             2512 non-null   object        
 1   account_id                 2512 non-null   object        
 2   transaction_amount         2512 non-null   float64       
 3   transaction_date           2512 non-null   datetime64[ns]
 4   transaction_type           2512 non-null   object        
 5   location                   2512 non-null   object        
 6   device_id                  2512 non-null   object        
 7   ip_address                 2512 non-null   object        
 8   merchant_id                2512 non-null   object        
 9   channel                    2512 non-null   object        
 10  customer_age               2512 non-null   int64         
 11  customer_occupation        2512 non-null   object        
 12  transa

In [5]:
df.describe()

,transaction_amount,transaction_date,customer_age,transaction_duration,login_attempts,account_balance,previous_transaction_date
count,2512.000000,2512,2512.000000,2512.000000,2512.000000,2512.000000,2512
mean,297.593778,2023-07-05 20:32:10.826433024,44.673965,119.643312,1.124602,5114.302966,2024-11-04 08:09:22.219745024
min,0.260000,2023-01-02 16:00:06,18.000000,10.000000,1.000000,101.250000,2024-11-04 08:06:23
25%,81.885000,2023-04-03 16:22:05.750000128,27.000000,63.000000,1.000000,1504.370000,2024-11-04 08:07:53
50%,211.140000,2023-07-07 17:49:43.500000,45.000000,112.500000,1.000000,4735.510000,2024-11-04 08:09:22
75%,414.527500,2023-10-06 18:40:53.500000,59.000000,161.000000,1.000000,7678.820000,2024-11-04 08:10:53.249999872
max,1919.110000,2024-01-01 18:21:50,80.000000,300.000000,5.000000,14977.990000,2024-11-04 08:12:23
std,291.946243,NaN,17.792198,69.963757,0.602662,3900.942499,NaN


#### Check 1: Finding Duplicates

In [8]:
dupe_count = df.duplicated(subset=['transaction_id']).sum()
print(f"Duplicate transaction_ids: {dupe_count}")

# Show the actual duplicate rows if any exist
dupes = df[df.duplicated(subset=['transaction_id'], keep=False)]
print(f"Rows involved in duplicates: {len(dupes)}")

print(dupes)



Duplicate transaction_ids: 0
Rows involved in duplicates: 0
Empty DataFrame
Columns: [transaction_id, account_id, transaction_amount, transaction_date, transaction_type, location, device_id, ip_address, merchant_id, channel, customer_age, customer_occupation, transaction_duration, login_attempts, account_balance, previous_transaction_date]
Index: []


#### Check 2 : Nulls and Empty Strings

In [9]:
null_counts = df.isnull().sum()
empty_counts = (df == '').sum()

profile = pd.DataFrame({
    'null_count': null_counts,
    'empty_count': empty_counts
})

print(profile)

                           null_count  empty_count
transaction_id                      0            0
account_id                          0            0
transaction_amount                  0            0
transaction_date                    0            0
transaction_type                    0            0
location                            0            0
device_id                           0            0
ip_address                          0            0
merchant_id                         0            0
channel                             0            0
customer_age                        0            0
customer_occupation                 0            0
transaction_duration                0            0
login_attempts                      0            0
account_balance                     0            0
previous_transaction_date           0            0


### Check 3: Data Standardisation and Consistency

In [11]:
df.describe(include='object')

,transaction_id,account_id,transaction_type,location,device_id,ip_address,merchant_id,channel,customer_occupation
count,2512,2512,2512,2512,2512,2512,2512,2512,2512
unique,2512,495,2,43,681,592,100,3,4
top,TX000001,AC00362,Debit,Fort Worth,D000697,200.136.146.93,M026,Branch,Student
freq,1,12,1944,70,9,13,45,868,657


In [22]:
cat_cols = ["transaction_type","channel","customer_occupation"]

for col in cat_cols:
    sum = 0
    print("\n")
    print(df[col].value_counts())
    sum += df[col].value_counts().values.sum()
    print("Total transactions: ", sum)




transaction_type
Debit     1944
Credit     568
Name: count, dtype: int64
Total transactions:  2512


channel
Branch    868
ATM       833
Online    811
Name: count, dtype: int64
Total transactions:  2512


customer_occupation
Student     657
Doctor      631
Engineer    625
Retired     599
Name: count, dtype: int64
Total transactions:  2512


In [21]:
inconsistent = df.groupby('account_id').agg(
    age_variations        = ('customer_age', 'nunique'),
    occupation_variations = ('customer_occupation', 'nunique')
)

problems = inconsistent[
    (inconsistent['age_variations'] > 1) |
    (inconsistent['occupation_variations'] > 1)
]

print(f"Accounts with inconsistent age or occupation: {len(problems)}")
print(problems)

Accounts with inconsistent age or occupation: 471
            age_variations  occupation_variations
account_id                                       
AC00001                  2                      2
AC00002                  7                      3
AC00003                  5                      3
AC00004                  9                      4
AC00005                  9                      4
...                    ...                    ...
AC00496                  3                      3
AC00497                  5                      3
AC00498                  8                      3
AC00499                  7                      4
AC00500                  4                      2

[471 rows x 2 columns]


### Check 4: Range & Outlier Checks

In [24]:
numeric_cols = ['transaction_amount', 'customer_age', 'transaction_duration', 'login_attempts', 'account_balance']
print(df[numeric_cols].describe())


       transaction_amount  customer_age  transaction_duration  login_attempts  \
count         2512.000000   2512.000000           2512.000000     2512.000000   
mean           297.593778     44.673965            119.643312        1.124602   
std            291.946243     17.792198             69.963757        0.602662   
min              0.260000     18.000000             10.000000        1.000000   
25%             81.885000     27.000000             63.000000        1.000000   
50%            211.140000     45.000000            112.500000        1.000000   
75%            414.527500     59.000000            161.000000        1.000000   
max           1919.110000     80.000000            300.000000        5.000000   

       account_balance  
count      2512.000000  
mean       5114.302966  
std        3900.942499  
min         101.250000  
25%        1504.370000  
50%        4735.510000  
75%        7678.820000  
max       14977.990000  


In [37]:
low_amount = df[df["transaction_amount"] < 1]
print(low_amount[['transaction_id','account_id','transaction_amount']])
print(f"Transactions under $1: {len(low_amount)}")

     transaction_id account_id  transaction_amount
490        TX000491    AC00166                0.99
929        TX000930    AC00329                0.86
1354       TX001355    AC00097                0.26
1879       TX001880    AC00460                0.84
2253       TX002254    AC00236                0.32
2259       TX002260    AC00324                0.45
Transactions under $1: 6


In [38]:
# High login attempts (more than 4 is suspicious in AML) (threshold setup hypothetically)
high_logins = df[df['login_attempts'] > 4]
print(f"High login attempts (>4): {len(high_logins)}")

High login attempts (>4): 32


In [ ]:
mean = df['transaction_amount'].mean()
std  = df['transaction_amount'].std()
high_amounts = df[df['transaction_amount'] > mean + 3*std]
print(f"Transaction amount outliers (3std): {len(high_amounts)}")
print(high_amounts[['transaction_id', 'account_id', 'transaction_amount']])

Transaction amount outliers (3std): 48
     transaction_id account_id  transaction_amount
74         TX000075    AC00265             1212.51
85         TX000086    AC00098             1340.19
176        TX000177    AC00363             1362.55
190        TX000191    AC00396             1422.55
274        TX000275    AC00454             1176.28
311        TX000312    AC00285             1221.65
340        TX000341    AC00107             1830.00
344        TX000345    AC00156             1271.90
375        TX000376    AC00316             1392.54
440        TX000441    AC00040             1237.56
475        TX000476    AC00464             1431.30
486        TX000487    AC00148             1416.69
535        TX000536    AC00161             1182.86
555        TX000556    AC00433             1282.86
614        TX000615    AC00466             1342.25
651        TX000652    AC00208             1241.05
653        TX000654    AC00423             1919.11
725        TX000726    AC00067             

In [39]:
high_value_txns  = df[df['transaction_amount'] > mean + 3*std].copy()
low_value_txns   = df[df['transaction_amount'] < 1].copy()
high_login_txns  = df[df['login_attempts'] > 3].copy()

### Check 5: Data Range Validation

In [47]:
# Earliset and latest transaction Dates

print(f"Earliest Transaction Date: {df["transaction_date"].min()}")
print(f"Latest Transaction Date: {df["transaction_date"].max()}")

# Previous > Forward Date

impossible_dates = df[df["previous_transaction_date"] > df['transaction_date']]
print(f"prev_transaction_date after transaction_date: {len(impossible_dates)}")

Earliest Transaction Date: 2023-01-02 16:00:06
Latest Transaction Date: 2024-01-01 18:21:50
prev_transaction_date after transaction_date: 2512


In [48]:
# Suspicious - is previous_transaction_date always the same value?
print(f"\nUnique previous_transaction_date values: {df['previous_transaction_date'].nunique()}")
print(df['previous_transaction_date'].value_counts().head(10))


Unique previous_transaction_date values: 360
previous_transaction_date
2024-11-04 08:09:17    16
2024-11-04 08:07:18    15
2024-11-04 08:11:10    15
2024-11-04 08:09:57    14
2024-11-04 08:12:18    13
2024-11-04 08:07:12    13
2024-11-04 08:07:24    13
2024-11-04 08:11:40    13
2024-11-04 08:06:59    13
2024-11-04 08:12:22    13
Name: count, dtype: int64


In [ ]:
### previous_transaction_date dropped — all values post-date actual transactions, confirmed to be dataset generation artifact not real banking data.

df = df.drop(columns=['previous_transaction_date'])
print(df.shape)

(2512, 15)
